In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from itertools import chain

In [0]:
def races_specific(df):
    df = df.drop("year")
    return df

def results_specific(df):
    df = df.withColumn("fastestlapspeed", F.col("fastestlapspeed").cast(DoubleType()))
    df = df.withColumn("fastestlap", F.col("fastestlap").cast(IntegerType()))
    return df

def safety_car_specific(df):
    df = df.withColumn("cause", 
        F.when(F.col("cause") == "multiple accidents", F.col("cause")) # Keep as is
            .when(F.col("cause").rlike("(?i)^accident.*"), "accident")        # Group all accident types
            .when(F.col("cause").rlike("(?i)^stranded car.*"), "stranded car") # Group all stranded types
            .otherwise(F.col("cause"))                                        # Keep everything else
    )
    return df

In [0]:
def apply_specific_silver_transforms(df, table_name ):
    # Dictionary mapping table names to their specific functions
    f1_transform_map = {
        "race_data_races": races_specific,
        "race_data_results": results_specific,
        "race_events_safety_car": safety_car_specific
    }
    
    # Look up the function in the dictionary
    if table_name in f1_transform_map:
        # Run the specific function and update df
        df = f1_transform_map[table_name](df)
        
    return df